# 02 - Nettoyage industriel et gouvernance

Ce notebook documente les transformations appliquees. Regle directrice : corriger les problemes de format, mais ne pas inventer une verite metier lorsque la source est contradictoire.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [ ]:
from src.preprocessing import (
    clean_campaigns, clean_destinations, clean_external, clean_market,
    clean_reviews, load_table, quality_summary
)

raw = ROOT / "data" / "raw"
destinations_raw = load_table(raw / "01_destinations_brut.csv")
market_raw = load_table(raw / "02_signaux_marche.xlsx")
reviews_raw = load_table(raw / "03_reviews_reseaux.json")
external_raw = load_table(raw / "04_facteurs_externes.csv")
campaigns_raw = load_table(raw / "06_campaign_history.json")

## 1. Diagnostic avant nettoyage

In [ ]:
raw_datasets = {
    "destinations_raw": destinations_raw,
    "market_raw": market_raw,
    "reviews_raw": reviews_raw,
    "external_raw": external_raw,
    "campaigns_raw": campaigns_raw,
}
pd.concat([quality_summary(df, name) for name, df in raw_datasets.items()], ignore_index=True)

Decision : les controles sont faits avant transformation pour garder une trace des anomalies sources. C'est important pour un jury : le nettoyage n'est pas une boite noire.

## 2. Application des fonctions de nettoyage

In [ ]:
destinations = clean_destinations(destinations_raw)
market = clean_market(market_raw)
reviews = clean_reviews(reviews_raw)
external = clean_external(external_raw)
campaigns = clean_campaigns(campaigns_raw)

cleaned_datasets = {
    "destinations": destinations,
    "market": market,
    "reviews": reviews,
    "external": external,
    "campaigns": campaigns,
}

for name, df in cleaned_datasets.items():
    print(f"{name}: {df.shape}")
    display(df.head(3))

## 3. Justification des transformations

In [ ]:
transformations = pd.DataFrame([
    ["country", "strip + title case", "Referentiel pays commun", "Evite pertes de jointure France/france/FRANCE"],
    ["destination_original", "conservation du code City_n", "Lineage gouvernance", "Permet d'auditer le mapping"],
    ["destination", "mapping vers nom de ville", "Restitution metier lisible", "Le mapping est documente dans src/city_mapping.py"],
    ["month", "conversion datetime", "Split temporel fiable", "Les dates invalides sont exclues"],
    ["campaign_budget", "parse numerique", "Calcul ROI/budget", "unknown reste manquant, pas impute arbitrairement"],
    ["status vs roi", "flag contradiction", "Gouvernance", "L'anomalie est exposee au lieu d'etre masquee"],
    ["reviews", "aggregation ulterieure", "Granularite destination", "Evite duplication artificielle"],
], columns=["Variable", "Transformation", "Justification metier", "Regle de prudence"])
transformations

## 4. Controle avant / apres

In [ ]:
before_after = []
for raw_name, clean_name in [
    ("destinations_raw", "destinations"),
    ("market_raw", "market"),
    ("reviews_raw", "reviews"),
    ("external_raw", "external"),
    ("campaigns_raw", "campaigns"),
]:
    raw_df = raw_datasets[raw_name]
    clean_df = cleaned_datasets[clean_name]
    before_after.append({
        "dataset": clean_name,
        "rows_before": len(raw_df),
        "rows_after": len(clean_df),
        "duplicates_before": raw_df.duplicated().sum(),
        "duplicates_after": clean_df.duplicated().sum(),
        "missing_before": raw_df.isna().sum().sum(),
        "missing_after": clean_df.isna().sum().sum(),
    })
pd.DataFrame(before_after)

In [ ]:
campaigns[campaigns["campaign_contradiction"] | campaigns["campaign_budget"].isna()]

Interpretation : les valeurs budget non numeriques et contradictions campagne sont conservees sous forme de signaux de qualite. Elles pourront penaliser ou limiter le scoring, mais elles ne doivent pas etre supprimees sans mandat metier.

## 5. Sauvegarde processed

In [ ]:
processed = ROOT / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)
for name, df in cleaned_datasets.items():
    df.to_csv(processed / f"{name}_clean.csv", index=False)
sorted(p.name for p in processed.glob("*_clean.csv"))